In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

In [3]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

sample = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

In [4]:
train.shape
test.shape

train.head()
test.head()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [5]:
train["answer"].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [6]:
train["prompt"].str.len()

0       142
1        43
2       182
3       129
4       110
       ... 
1995     72
1996    104
1997     93
1998    266
1999    140
Name: prompt, Length: 2000, dtype: int64

In [8]:
train["prompt"].str.split().apply(len)

0       22
1        5
2       27
3       19
4       15
        ..
1995    10
1996    16
1997    13
1998    37
1999    18
Name: prompt, Length: 2000, dtype: int64

In [11]:
idx = train["prompt"].str.len().idxmax()

In [13]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.metrics.pairwise import cosine_similarity



# Q1
# Frequency distribution of correct answers
# Sum of most frequent + least frequent


answer_counts = train["answer"].value_counts().sort_index()

print("\nQ1 Frequency Distribution")
print(answer_counts)

q1 = answer_counts.max() + answer_counts.min()

print("\nQ1 Answer:")
print(q1)


# Q2
# Lowercase + remove punctuation
# Vocabulary size of prompt column


translator = str.maketrans("", "", string.punctuation)

clean_prompts = (
    train["prompt"]
    .fillna("")
    .str.lower()
    .apply(lambda x: x.translate(translator))
)

vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

q2 = len(vocab)

print("\nQ2 Vocabulary Size:")
print(q2)


# Q3
# Row ID 1
# Remove sklearn English stopwords
# Count remaining words


row1_prompt = clean_prompts.iloc[0]

words = row1_prompt.split()

filtered_words = [
    w for w in words
    if w not in ENGLISH_STOP_WORDS
]

q3 = len(filtered_words)

print("\nQ3 Remaining Words:")
print(q3)


# Q4
# TFIDF on combined text
# Vocabulary size

combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")

vectorizer.fit(combined_text)

q4 = len(vectorizer.vocabulary_)

print("\nQ4 TFIDF Feature Count:")
print(q4)


# Q5
# Cosine similarity between prompt and option A



prompt_vec = vectorizer.transform([train.loc[0, "prompt"]])
A_vec = vectorizer.transform([train.loc[0, "A"]])

q5 = cosine_similarity(prompt_vec, A_vec)[0, 0]

print("\nQ5 Similarity Prompt vs A (Row1):")
print(round(q5, 4))


# Q6
# Accuracy %


option_cols = ["A", "B", "C", "D", "E"]

correct = 0

for _, row in train.iterrows():

    prompt_text = str(row["prompt"])

    prompt_vec = vectorizer.transform([prompt_text])

    sims = []

    for opt in option_cols:

        option_vec = vectorizer.transform([str(row[opt])])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0, 0]

        sims.append(sim)

    predicted = option_cols[np.argmax(sims)]

    if predicted == row["answer"]:
        correct += 1

q6 = 100 * correct / len(train)

print("\nQ6 Highest Similarity Accuracy (%):")
print(round(q6, 4))


# Q7
# MAP@3


truth = "C"
preds = ["C", "A", "B"]

if truth in preds:
    rank = preds.index(truth) + 1
    q7 = 1 / rank
else:
    q7 = 0

print("\nQ7 MAP@3:")
print(q7)


# Q8
# MAP@3


truth = "B"
preds = ["D", "B", "E"]

if truth in preds:
    rank = preds.index(truth) + 1
    q8 = 1 / rank
else:
    q8 = 0

print("\nQ8 MAP@3:")
print(q8)


# MAP@3 helper


def map3_single(actual, preds):
    if actual in preds[:3]:
        return 1 / (preds[:3].index(actual) + 1)
    return 0


# Q9
# Majority Class Baseline


freq_order = train["answer"].value_counts().index.tolist()

baseline_preds = freq_order[:3]

scores = [
    map3_single(ans, baseline_preds)
    for ans in train["answer"]
]

q9 = np.mean(scores)

print("\nQ9 Majority Class MAP@3:")
print(q9)


# Q10
# TFIDF Pipeline MAP@3


all_scores = []

for _, row in train.iterrows():

    prompt_text = str(row["prompt"])

    prompt_vec = vectorizer.transform([prompt_text])

    similarities = {}

    for opt in option_cols:

        option_vec = vectorizer.transform([str(row[opt])])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0, 0]

        similarities[opt] = sim

    ranked = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    score = map3_single(
        row["answer"],
        ranked[:3]
    )

    all_scores.append(score)

q10 = np.mean(all_scores)

print("\nQ10 TFIDF Pipeline MAP@3:")
print(q10)



Q1 Frequency Distribution
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Q1 Answer:
814

Q2 Vocabulary Size:
859

Q3 Remaining Words:
13

Q4 TFIDF Feature Count:
2762

Q5 Similarity Prompt vs A (Row1):
0.272

Q6 Highest Similarity Accuracy (%):
13.55

Q7 MAP@3:
1.0

Q8 MAP@3:
0.5

Q9 Majority Class MAP@3:
0.42125

Q10 TFIDF Pipeline MAP@3:
0.2961666666666667
